In [ ]:
# Loan-Level Data

select
b.BLOCK_TIMESTAMP,
b.TX_HASH,
b.EVENT_NAME,
b.ORIGIN_FROM_ADDRESS,
b.ORIGIN_TO_ADDRESS,
b.PLATFORM,
b.BORROWER,
b.TOKEN_ADDRESS,
b.TOKEN_SYMBOL,
m.SYMBOL,
m.NAME,
m.DECIMALS,
b.AMOUNT_UNADJ,
b.AMOUNT,
b.AMOUNT_USD
from base.defi.ez_lending_borrows as b
left join base.price.ez_asset_metadata as m
ON b.TOKEN_ADDRESS = m.TOKEN_ADDRESS
where b.BLOCK_TIMESTAMP between ('2024-01-01') and ('2024-03-01')
and PLATFORM = 'Aave V3' ;

In [ ]:
# 📊 Data Handling
import pandas as pd
import numpy as np
import requests

# 📈 Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# ⚙️ Preprocessing & Scaling
from datetime import timedelta
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

# 📈 Analysis
import scipy.stats as stats
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
  
# 🧠 Modeling
from sklearn.cluster import KMeans, DBSCAN
import hdbscan
from sklearn.metrics import silhouette_score

## 0: Import data

In [ ]:
# brrows
borrows_df = pd.read_csv('01_borrows_data.csv',on_bad_lines='skip')
# deposits
deposits_df = pd.read_csv('deposits.csv',on_bad_lines='skip')
# liquidations
lq_df = pd.read_csv('liquidations.csv',on_bad_lines='skip')
# repayments
py_df = pd.read_csv('repayments.csv',on_bad_lines='skip')

In [ ]:
deposits_df.head(1)

In [ ]:
# Aggregate daily positions per user

deposits_df['BLOCK_TIMESTAMP'] = pd.to_datetime(deposits_df['BLOCK_TIMESTAMP']).dt.date
deposits_daily = deposits_df.groupby(['DEPOSITOR', 'BLOCK_TIMESTAMP', 'TOKEN_SYMBOL']).agg({'AMOUNT_USD':'sum'}).reset_index()
deposits_daily = deposits_daily.rename(columns={'DEPOSITOR': 'user_address'})
deposits_daily = deposits_daily.rename(columns={'AMOUNT_USD': 'total_supplied_usd'})


borrows_df['BLOCK_TIMESTAMP'] = borrows_df['BLOCK_TIMESTAMP'].astype(str).str.replace(r"[^\x00-\x7F]+", "", regex=True)
borrows_df['BLOCK_TIMESTAMP'] = pd.to_datetime(borrows_df['BLOCK_TIMESTAMP'], errors='coerce')
borrows_df['BLOCK_TIMESTAMP'] = borrows_df['BLOCK_TIMESTAMP'].dt.date


borrows_daily = borrows_df.groupby(['BORROWER', 'BLOCK_TIMESTAMP', 'TOKEN_SYMBOL']).agg({'AMOUNT_USD':'sum'}).reset_index()
borrows_daily = borrows_daily.rename(columns={'BORROWER': 'user_address'})
borrows_daily = borrows_daily.rename(columns={'AMOUNT_USD': 'total_borrowed_usd'})


merged_df = pd.merge(
    deposits_daily,
    borrows_daily,
    on=['user_address', 'BLOCK_TIMESTAMP', 'TOKEN_SYMBOL'],
    how='outer').fillna(0)

In [ ]:
merged_df['TOKEN_SYMBOL'].value_counts()

In [ ]:
merged_df.head()

In [ ]:
# Calculate Daily Portfolio Summary

merged_df['BLOCK_TIMESTAMP'] = pd.to_datetime(merged_df['BLOCK_TIMESTAMP']).dt.date

# Group by user and day to sum across tokens
portfolio_df = (
    merged_df.groupby(['user_address', 'BLOCK_TIMESTAMP'])
    .agg(
        total_collateral_usd=('total_supplied_usd', 'sum'),
        total_debt_usd=('total_borrowed_usd', 'sum')
    )
    .reset_index()
)

In [ ]:
portfolio_df

In [ ]:
# Calculate Health Factor and Simulate Shocks

# Step 1: Estimate Health Factor Proxy
portfolio_df['health_factor'] = np.where(
    portfolio_df['total_debt_usd'] == 0,
    np.inf,
    0.85 * portfolio_df['total_collateral_usd'] / portfolio_df['total_debt_usd']
)

# Step 2: Simulate price shocks (e.g., collateral drops by 10%, 20%, ..., 90%)
shock_levels = np.linspace(0.1, 0.9, 9)  # 10% to 90%

# Create one row per user per shock level
simulation = []

for shock in shock_levels:
    shocked_collateral = (1 - shock) * portfolio_df['total_collateral_usd']
    shocked_hf = np.where(
        portfolio_df['total_debt_usd'] == 0,
        np.inf,
        0.85 * shocked_collateral / portfolio_df['total_debt_usd']
    )
    prob_default = (shocked_hf < 1).astype(int)

    for i in range(len(portfolio_df)):
        simulation.append({
            'user_address': portfolio_df.loc[i, 'user_address'],
            'date': portfolio_df.loc[i, 'BLOCK_TIMESTAMP'],
            'shock_pct': shock,
            'shocked_health_factor': shocked_hf[i],
            'will_liquidate': prob_default[i]
        })

# Convert to DataFrame
simulation_df = pd.DataFrame(simulation)


In [ ]:
simulation_df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming you already have this DataFrame ready:
# simulation_df with columns: ['shock_pct', 'will_liquidate']

# Step 1: Aggregate percentage of users who will liquidate at each shock level
summary = simulation_df.groupby('shock_pct')['will_liquidate'].mean().reset_index()
summary['percent_liquidated'] = summary['will_liquidate'] * 100

# Step 2: Plot
plt.figure(figsize=(10, 6))
sns.lineplot(
    data=summary,
    x='shock_pct',
    y='percent_liquidated',
    marker='o',
    color='orange'
)

# Formatting
plt.title('Liquidation Risk vs. Collateral Price Shock')
plt.xlabel('Collateral Price Drop (%)')
plt.ylabel('Users at Risk of Liquidation (%)')
plt.xticks(summary['shock_pct'], [f"{int(s*100)}%" for s in summary['shock_pct']])
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# مقدار وثیقه تا زمان وام برای هر کاربر
initial_collateral = deposits_df.groupby("DEPOSITOR")["AMOUNT_USD"].sum().rename("INITIAL_COLLATERAL")

# مقدار بدهی اولیه از borrow
initial_debt = borrows_df.groupby("BORROWER")["AMOUNT_USD"].sum().rename("INITIAL_BORROWED")

liq_threshold = 0.85
# ادغام و محاسبه
initial_df = pd.concat([initial_collateral, initial_debt], axis=1).fillna(0)
initial_df["INITIAL_HEALTH_FACTOR"] = (initial_df["INITIAL_COLLATERAL"] * liq_threshold) / initial_df["INITIAL_BORROWED"]
initial_df["INITIAL_HEALTH_FACTOR"] = initial_df["INITIAL_HEALTH_FACTOR"].replace([np.inf, -np.inf], 0).fillna(0)

# ــــ
import numpy as np
import pandas as pd

# فرض: price_df شامل ستون‌های 'timestamp', 'token', 'price_usd'
price_df["log_return"] = np.log(price_df["price_usd"] / price_df["price_usd"].shift(1))
volatility_df = price_df.groupby("token")["log_return"].std().rename("VOLATILITY_ESTIMATE")


# Example input (replace this with your real DataFrame)
user_df = pd.DataFrame({
    'user_address': [f"user_{i}" for i in range(100)],
    'initial_health_factor': np.random.uniform(1.05, 1.3, size=100),
    'volatility_estimate': np.random.uniform(0.03, 0.1, size=100)
})

# Simulation parameters
n_days = 30
dt = 1  # Daily intervals
mu = 0  # Assume zero drift (can customize per asset)

# Create matrix to store health factor paths
hf_paths = np.zeros((len(user_df), n_days + 1))

# Set initial HF values
hf_paths[:, 0] = user_df['initial_health_factor'].values

# Simulate GBM for each user
for t in range(1, n_days + 1):
    z = np.random.standard_normal(len(user_df))
    sigma = user_df['volatility_estimate'].values
    hf_paths[:, t] = hf_paths[:, t - 1] * np.exp((mu - 0.5 * sigma ** 2) * dt + sigma * np.sqrt(dt) * z)

# Plot
plt.figure(figsize=(12, 6))
for i in range(len(user_df)):
    plt.plot(hf_paths[i], lw=0.8, alpha=0.6)

plt.axhline(y=1.0, color='red', linestyle='--', label='Liquidation Threshold (HF=1)')
plt.title("Simulated Health Factor Paths")
plt.xlabel("Days")
plt.ylabel("Health Factor")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Determine how many times each user crosses HF=1 during simulation
# Step 1: Pick the most recent snapshot per user
latest_df = portfolio_df.sort_values('date').groupby('user_address').tail(1)

# Step 2: Filter for users who have active debt (health factor < inf)
latest_df = latest_df[latest_df['total_debt_usd'] > 0]

# Step 3: Create user_df
user_df = latest_df[['user_address', 'health_factor']].copy()
user_df.rename(columns={'health_factor': 'initial_health_factor'}, inplace=True)

# Step 4: Assign a constant volatility estimate for now (e.g. 5%)
user_df['volatility_estimate'] = 0.05  # or vary this later per asset






n_paths = hf_paths.shape[0]
n_days = hf_paths.shape[1]

# Binary matrix: 1 if HF < 1 at that time step
liquidation_events = (hf_paths < 1).astype(int)

# Calculate if user ever hits liquidation threshold
ever_liquidated = liquidation_events.max(axis=1)  # 1 if ever HF < 1

# Result table
user_df['liquidation_probability'] = ever_liquidated


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(user_profile_df["REPAYMENT_RATIO"], bins=30, kde=True, color="skyblue")
plt.title("Distribution of Repayment Ratios")
plt.xlabel("Repayment Ratio")
plt.ylabel("Number of Users")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x="NUM_LIQUIDATIONS", data=user_profile_df, color="salmon")
plt.title("Number of Liquidations per User")
plt.xlabel("Liquidation Count")
plt.ylabel("Number of Users")
plt.xticks(rotation=0)
plt.grid(axis='y')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    x="TOTAL_BORROWED",
    y="REPAYMENT_RATIO",
    hue="HIGH_RISK_FLAG",
    data=user_profile_df,
    palette={0: "green", 1: "red"},
    alpha=0.7
)
plt.title("Borrowed Amount vs Repayment Ratio (colored by risk)")
plt.xlabel("Total Borrowed (USD)")
plt.ylabel("Repayment Ratio")
plt.xscale("log")
plt.grid(True)
plt.show()

In [ ]:
risk_counts = user_profile_df["HIGH_RISK_FLAG"].value_counts()
labels = ["Low Risk", "High Risk"]
colors = ["lightgreen", "tomato"]

plt.figure(figsize=(6, 6))
plt.pie(risk_counts, labels=labels, autopct='%1.1f%%', colors=colors, startangle=140)
plt.title("User Risk Classification")
plt.axis('equal')
plt.show()

# 4.3 Liquidation Event Modeling

In [ ]:
# فرض: df شامل ستون‌های زیر هست:
# - COLLATERAL_AMOUNT
# - COLLATERAL_PRICE_USD
# - LOAN_AMOUNT_USD
# - LIQUIDATION_THRESHOLD

# مرحله 1: محاسبه ارزش وثیقه
df["COLLATERAL_VALUE_USD"] = df["COLLATERAL_AMOUNT"] * df["COLLATERAL_PRICE_USD"]

# مرحله 2: محاسبه Health Factor
df["HEALTH_FACTOR"] = (
    df["COLLATERAL_VALUE_USD"] * df["LIQUIDATION_THRESHOLD"]
) / df["LOAN_AMOUNT_USD"]

# مرحله 3: پرچم گذاری ریسک
df["IS_AT_RISK"] = (df["HEALTH_FACTOR"] < 1.0).astype(int)

# نمایش
df[["COLLATERAL_VALUE_USD", "HEALTH_FACTOR", "IS_AT_RISK"]].head()

In [ ]:
# brrows
br_df = pd.read_csv('01_borrows_data.csv',on_bad_lines='skip')
# deposits
depo_df = pd.read_csv('deposits.csv',on_bad_lines='skip')
# liquidations
lq_df = pd.read_csv('liquidations.csv',on_bad_lines='skip')
# repayments
py_df = pd.read_csv('repayments.csv',on_bad_lines='skip')

In [ ]:
br_df.head()

In [ ]:
br_df['TOKEN_SYMBOL'].value_counts()

In [ ]:
liq_threshold_map = {
    "WETH": 0.825,
    "USDbC": 0.9,
    "USDC": 0.86,
    "cbETH": 0.77,
    "wstETH": 0.81}

br_df["LIQUIDATION_THRESHOLD"] = br_df["TOKEN_SYMBOL"].map(liq_threshold_map)
br_df[["TOKEN_SYMBOL", "LIQUIDATION_THRESHOLD"]].drop_duplicates()

In [ ]:
# محاسبه Collateral USD برای هر کاربر
collateral_df = depo_df.groupby("DEPOSITOR")["AMOUNT_USD"].sum().rename("TOTAL_COLLATERAL")

# الحاق به دیتافریم نهایی کاربران
user_profile_df = user_profile_df.join(collateral_df, how="left")
user_profile_df["TOTAL_COLLATERAL"] = user_profile_df["TOTAL_COLLATERAL"].fillna(0)

# میانگین آستانه لیکوئیدیشن برای هر کاربر (بر اساس وام‌ها)
liq_thresh_per_user = br_df.groupby("BORROWER")["LIQUIDATION_THRESHOLD"].mean().rename("AVG_LIQ_THRESHOLD")
user_profile_df = user_profile_df.join(liq_thresh_per_user, how="left")


# فرمول اصلی
user_profile_df["HEALTH_FACTOR"] = (
    user_profile_df["TOTAL_COLLATERAL"] * user_profile_df["AVG_LIQ_THRESHOLD"]
) / user_profile_df["TOTAL_BORROWED"]

# جلوگیری از تقسیم بر صفر
user_profile_df["HEALTH_FACTOR"] = user_profile_df["HEALTH_FACTOR"].replace([np.inf, -np.inf], np.nan).fillna(0)

user_profile_df[["TOTAL_BORROWED", "TOTAL_COLLATERAL", "AVG_LIQ_THRESHOLD", "HEALTH_FACTOR"]].head()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(user_profile_df["HEALTH_FACTOR"], bins=30, kde=True, color="skyblue")
plt.title("Distribution of Health Factor")
plt.xlabel("Health Factor")
plt.ylabel("Number of Users")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
user_profile_df["HF_10_DROP"] = (user_profile_df["TOTAL_COLLATERAL"] * 0.9 * user_profile_df["AVG_LIQ_THRESHOLD"]) / user_profile_df["TOTAL_BORROWED"]
user_profile_df["HF_30_DROP"] = (user_profile_df["TOTAL_COLLATERAL"] * 0.7 * user_profile_df["AVG_LIQ_THRESHOLD"]) / user_profile_df["TOTAL_BORROWED"]
# و الی آخر

In [ ]:
user_profile_df

In [ ]:
# میانگین مقدار وثیقه و آستانه لیکویید شدن برای هر کاربر
collateral_df = depo_df.groupby("DEPOSITOR").agg({
    "AMOUNT_USD": "sum",
    "TOKEN_SYMBOL": lambda x: list(x.unique())  # برای بررسی نوع دارایی‌ها
}).rename(columns={"AMOUNT_USD": "TOTAL_COLLATERAL"})

# تعریف آستانه‌های لیکوئید شدن به صورت دیکشنری
liq_thresholds = {
    "WETH": 0.825,
    "cbETH": 0.77,
    "wstETH": 0.75,
    "USDbC": 0.90,
    "USDC": 0.90,
    # در صورت نیاز می‌تونی توکن‌های دیگه اضافه کنی
}

# محاسبه میانگین LIQ_THRESHOLD برای کاربرانی که چند نوع توکن دارند
def avg_liq_thresh(token_list):
    valid = [liq_thresholds[tok] for tok in token_list if tok in liq_thresholds]
    return sum(valid) / len(valid) if valid else 0.75  # مقدار پیش‌فرض

collateral_df["AVG_LIQ_THRESHOLD"] = collateral_df["TOKEN_SYMBOL"].apply(avg_liq_thresh)

# merge با user_profile_df
user_profile_df = user_profile_df.merge(collateral_df[["TOTAL_COLLATERAL", "AVG_LIQ_THRESHOLD"]],
                                         left_index=True, right_index=True, how="left").fillna(0)

In [ ]:
user_profile_df

In [ ]:
shock_percent = 0.30
shock_multiplier = 1 - shock_percent

# محاسبه Health Factor جدید بعد از شوک
user_profile_df["HF_SHOCKED"] = (
    user_profile_df["TOTAL_BORROWED"] * shock_multiplier * user_profile_df["AVG_LIQ_THRESHOLD_x"]
) / user_profile_df["TOTAL_BORROWED"]

# جلوگیری از تقسیم بر صفر و نان
user_profile_df["HF_SHOCKED"] = user_profile_df["HF_SHOCKED"].replace([np.inf, -np.inf], np.nan).fillna(0)

# پرچم‌گذاری کاربران در آستانه لیکوئید شدن
user_profile_df["LIKELY_TO_BE_LIQUIDATED"] = (user_profile_df["HF_SHOCKED"] < 1).astype(int)

# نمایش آماری
num_at_risk = user_profile_df["LIKELY_TO_BE_LIQUIDATED"].sum()
total_users = user_profile_df.shape[0]
risk_percent = 100 * num_at_risk / total_users

print(f"📉 تعداد کاربرانی که در صورت کاهش 30٪ قیمت وثیقه، در آستانه لیکوئید شدن قرار می‌گیرند: {num_at_risk} از {total_users} ({risk_percent:.2f}%)")